In [1]:
import pandas as pd
import numpy as np

In [4]:
# Load the Titanic dataset
df = pd.read_csv('train.csv')
print("Dataset loaded successfully!")
print(f"Shape of the dataset: {df.shape}")
df.head()

Dataset loaded successfully!
Shape of the dataset: (891, 12)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [6]:
# Step 1: Check for duplicate rows based on all columns
print("=" * 50)
print("CHECKING FOR DUPLICATES")
print("=" * 50)

# Count the number of duplicate rows
num_duplicates = df.duplicated().sum()
print(f"\nNumber of duplicate rows (based on all columns): {num_duplicates}")

# Display the duplicate rows if any exist
if num_duplicates > 0:
    print(f"\nShowing duplicate rows (first 10):")
    print(df[df.duplicated(keep=False)].sort_values(by=list(df.columns)).head(10))
else:
    print("\nNo duplicate rows found in the dataset.")

CHECKING FOR DUPLICATES

Number of duplicate rows (based on all columns): 0

No duplicate rows found in the dataset.


In [ ]:
# Step 2: Store the original shape
original_shape = df.shape
print("\n" + "=" * 50)
print("REMOVING DUPLICATES")
print("=" * 50)
print(f"\nOriginal dataset shape: {original_shape}")
print(f"Original number of rows: {original_shape[0]}")

# Remove duplicate rows (keeping the first occurrence)
df_cleaned = df.drop_duplicates()

# Store the cleaned shape
cleaned_shape = df_cleaned.shape
print(f"\nCleaned dataset shape: {cleaned_shape}")
print(f"Cleaned number of rows: {cleaned_shape[0]}")

# Calculate the difference
rows_removed = original_shape[0] - cleaned_shape[0]
print(f"\nNumber of duplicate rows removed: {rows_removed}")

In [ ]:
# Step 3: Verify the removal of duplicates
print("\n" + "=" * 50)
print("VERIFICATION")
print("=" * 50)

# Check if there are any duplicates left in the cleaned dataset
remaining_duplicates = df_cleaned.duplicated().sum()
print(f"\nNumber of duplicates in cleaned dataset: {remaining_duplicates}")

if remaining_duplicates == 0:
    print("✓ Successfully removed all duplicate rows!")
else:
    print(f"⚠ Warning: {remaining_duplicates} duplicate rows still remain.")

# Summary
print("\n" + "=" * 50)
print("SUMMARY")
print("=" * 50)
print(f"Before: {original_shape[0]} rows")
print(f"After: {cleaned_shape[0]} rows")
print(f"Removed: {rows_removed} duplicate rows")
print(f"Percentage removed: {(rows_removed/original_shape[0]*100):.2f}%")

In [ ]:
# ============================================================
# Exercise 2: Handling Missing Values — Titanic Dataset
# ============================================================

import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer

# ── Load dataset ─────────────────────────────────────────────
df = pd.read_csv('train.csv')
print(f"Dataset shape: {df.shape}\n")

# ============================================================
# STEP 1: Identify columns with missing values
# ============================================================
missing_count = df.isnull().sum()
missing_pct   = (df.isnull().sum() / len(df) * 100).round(2)

missing_summary = pd.DataFrame({
    'Missing Count': missing_count,
    'Missing %':     missing_pct,
    'Dtype':         df.dtypes
})
cols_with_missing = missing_summary[missing_summary['Missing Count'] > 0]

print("=" * 55)
print("STEP 1: Columns with Missing Values")
print("=" * 55)
print(cols_with_missing)
# Output:
#           Missing Count  Missing %   Dtype
# Age                 177      19.87   float64
# Cabin               687      77.10   object
# Embarked              2       0.22   object

# ============================================================
# STRATEGY 1: Removal — Drop 'Cabin' column
# ============================================================
# Rationale: 77.1% of values are missing. Imputing over ¾ of
# a column introduces more noise than signal, so it is safer
# to drop it entirely.
# ─────────────────────────────────────────────────────────────
print("\n" + "=" * 55)
print("STRATEGY 1: REMOVAL — Drop 'Cabin' column")
print("Reason: 77.1% missing — too sparse to impute reliably")
print("=" * 55)

df_clean = df.copy()
df_clean.drop(columns=['Cabin'], inplace=True)

print(f"  Shape before : {df.shape}")           # (891, 12)
print(f"  Shape after  : {df_clean.shape}")     # (891, 11)

# ============================================================
# STRATEGY 2: Median Imputation — 'Age' via SimpleImputer
# ============================================================
# Rationale: Age is numerical and moderately skewed (mean 29.7,
# std 14.5). The median (28.0) is more robust to outliers than
# the mean. SimpleImputer makes this easy and pipeline-friendly.
# ─────────────────────────────────────────────────────────────
print("\n" + "=" * 55)
print("STRATEGY 2: MEDIAN IMPUTATION — 'Age' (SimpleImputer)")
print("Reason: Numeric + skewed → median is robust to outliers")
print("=" * 55)

imputer = SimpleImputer(strategy='median')
df_clean['Age'] = imputer.fit_transform(df_clean[['Age']])
median_used = imputer.statistics_[0]

print(f"  Missing before : {df['Age'].isnull().sum()}")   # 177
print(f"  Missing after  : {df_clean['Age'].isnull().sum()}")  # 0
print(f"  Median used    : {median_used}")                # 28.0
print(f"  Mean before    : {df['Age'].mean():.4f}")
print(f"  Mean after     : {df_clean['Age'].mean():.4f}")

# ============================================================
# STRATEGY 3: Mode Fill — 'Embarked' via fillna()
# ============================================================
# Rationale: Only 2 values are missing (0.22%). 'Embarked' is
# a categorical port code (S/C/Q). Filling with the mode ('S'
# = Southampton, 72% of passengers) is a safe, minimal choice.
# ─────────────────────────────────────────────────────────────
print("\n" + "=" * 55)
print("STRATEGY 3: MODE FILL — 'Embarked' (fillna)")
print("Reason: Categorical + only 2 missing → use most frequent")
print("=" * 55)

mode_embarked = df_clean['Embarked'].mode()[0]          # 'S'
df_clean['Embarked'] = df_clean['Embarked'].fillna(mode_embarked)

print(f"  Missing before : {df['Embarked'].isnull().sum()}")  # 2
print(f"  Missing after  : {df_clean['Embarked'].isnull().sum()}")  # 0
print(f"  Mode used      : '{mode_embarked}' (Southampton)")
print(f"  Value counts   :\n{df_clean['Embarked'].value_counts().to_string()}")

# ============================================================
# FINAL: Verify no missing values remain
# ============================================================
print("\n" + "=" * 55)
print("FINAL: Missing Value Summary After All Strategies")
print("=" * 55)

remaining = df_clean.isnull().sum()
if remaining.sum() == 0:
    print("  ✓ No missing values remain!")
else:
    print(remaining[remaining > 0])

print(f"\n  Final dataset shape: {df_clean.shape}")
print("\n  Preview (selected columns):")
print(df_clean[['PassengerId', 'Survived', 'Pclass', 'Age', 'Embarked']].head())

In [ ]:
# ============================================================
# Exercise 3: Feature Engineering — Titanic Dataset
# ============================================================

import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder

# ── Load & apply Exercise 2 cleaning ────────────────────────
df = pd.read_csv('train.csv')
df.drop(columns=['Cabin'], inplace=True)
df['Age'] = SimpleImputer(strategy='median').fit_transform(df[['Age']])
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])

# ============================================================
# STEP 1: Create FamilySize & IsAlone
# ============================================================
# FamilySize = siblings/spouses + parents/children + self
# IsAlone    = 1 if travelling solo, 0 otherwise
# ─────────────────────────────────────────────────────────────
print("=" * 55)
print("STEP 1: NEW FEATURES — FamilySize & IsAlone")
print("=" * 55)

df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
df['IsAlone']    = (df['FamilySize'] == 1).astype(int)

print(df[['SibSp', 'Parch', 'FamilySize', 'IsAlone']].head(8).to_string(index=False))
print(f"\n  FamilySize range : {df['FamilySize'].min()} – {df['FamilySize'].max()}")
print(f"  Travelling alone : {df['IsAlone'].sum()} passengers ({df['IsAlone'].mean()*100:.1f}%)")

# ============================================================
# STEP 2: Extract Title from Name
# ============================================================
# Regex captures the title between ", " and "." in the Name.
# Rare/foreign titles are grouped into broader categories to
# avoid sparse one-hot columns later.
# ─────────────────────────────────────────────────────────────
print("\n" + "=" * 55)
print("STEP 2: NEW FEATURE — Title (extracted from Name)")
print("=" * 55)

df['Title'] = df['Name'].str.extract(r',\s*([^\.]+)\.')

title_map = {
    'Mr':     'Mr',
    'Miss':   'Miss',
    'Mrs':    'Mrs',
    'Master': 'Master',
    # Military / professional → Officer
    'Dr': 'Officer', 'Rev': 'Officer', 'Major': 'Officer',
    'Col': 'Officer', 'Capt': 'Officer',
    # Equivalent feminine titles → normalised
    'Mlle': 'Miss', 'Ms': 'Miss', 'Mme': 'Mrs',
    # Nobility → Royalty
    'Lady': 'Royalty', 'Sir': 'Royalty',
    'Don': 'Royalty', 'the Countess': 'Royalty', 'Jonkheer': 'Royalty'
}
df['Title'] = df['Title'].map(title_map)

print("  Title value counts:")
print(df['Title'].value_counts().to_string())
# Mr: 517 | Miss: 185 | Mrs: 126 | Master: 40 | Officer: 18 | Royalty: 5

# ============================================================
# STEP 3: One-Hot Encoding — 'Sex' & 'Embarked'
# ============================================================
# Best for nominal categories with no ordinal relationship.
# pd.get_dummies adds one binary column per category.
# drop_first=False keeps all columns for interpretability
# (can set True to avoid multicollinearity in linear models).
# ─────────────────────────────────────────────────────────────
print("\n" + "=" * 55)
print("STEP 3: ONE-HOT ENCODING — Sex & Embarked")
print("=" * 55)

df = pd.get_dummies(df, columns=['Sex', 'Embarked'], drop_first=False)

sex_cols      = [c for c in df.columns if c.startswith('Sex_')]
embarked_cols = [c for c in df.columns if c.startswith('Embarked_')]
print(f"  Sex columns      : {sex_cols}")
print(f"  Embarked columns : {embarked_cols}")
print("\n  Sample:")
print(df[sex_cols + embarked_cols].head(5).to_string(index=False))

# ============================================================
# STEP 4: Label Encoding — 'Title'
# ============================================================
# Title has 6 grouped categories. LabelEncoder assigns each
# an integer (0–5). Suitable here because tree-based models
# (Random Forest, XGBoost) handle ordinal integers well.
# For linear models, one-hot encoding Title would be safer.
# ─────────────────────────────────────────────────────────────
print("\n" + "=" * 55)
print("STEP 4: LABEL ENCODING — Title")
print("=" * 55)

le = LabelEncoder()
df['Title_encoded'] = le.fit_transform(df['Title'])

mapping = dict(zip(le.classes_, le.transform(le.classes_)))
print(f"  Mapping : {mapping}")
# {'Master': 0, 'Miss': 1, 'Mr': 2, 'Mrs': 3, 'Officer': 4, 'Royalty': 5}

print("\n  Unique values:")
print(df[['Title', 'Title_encoded']].drop_duplicates()
        .sort_values('Title_encoded').to_string(index=False))

# ============================================================
# STEP 5: One-Hot Encoding — 'Pclass'
# ============================================================
# Pclass is 1/2/3 but is ordinal in meaning, not a true
# continuous number. Encoding as dummies prevents the model
# from assuming equal spacing (1→2 same as 2→3).
# ─────────────────────────────────────────────────────────────
print("\n" + "=" * 55)
print("STEP 5: ONE-HOT ENCODING — Pclass")
print("=" * 55)

df = pd.get_dummies(df, columns=['Pclass'], prefix='Pclass', drop_first=False)

pclass_cols = [c for c in df.columns if c.startswith('Pclass_')]
print(f"  Pclass columns: {pclass_cols}")
print("\n  Sample:")
print(df[pclass_cols].head(5).to_string(index=False))

# ============================================================
# FINAL: Drop raw / non-informative columns
# ============================================================
drop_cols = ['Name', 'Ticket', 'Title', 'PassengerId']
df_final  = df.drop(columns=drop_cols)

print("\n" + "=" * 55)
print("FINAL: Engineered Dataset Summary")
print("=" * 55)
print(f"  Shape: {df_final.shape}\n")
print(f"  {'Column':<25} {'Dtype'}")
print(f"  {'-'*35}")
for col in df_final.columns:
    print(f"  {col:<25} {str(df_final[col].dtype)}")
print(f"\n  Missing values: {df_final.isnull().sum().sum()}")
print("\n  First 3 rows:")
print(df_final.head(3).to_string())